In [17]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# 1. Load datasets
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 1   # Fake
true["label"] = 0   # Real

news_dataset = pd.concat([fake, true], axis=0)
news_dataset = news_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Clean data
news_dataset = news_dataset.fillna('')
news_dataset['content'] = news_dataset['text']   # only text (removes easy signals)

# 3. Preprocessing
bias_words = {"reuters", "said", "new", "year"}

def preprocess(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    
    words = [
        w for w in words
        if w not in ENGLISH_STOP_WORDS
        and w not in bias_words
        and len(w) > 2
    ]
    
    return ' '.join(words)

news_dataset['content'] = news_dataset['content'].apply(preprocess)

# 4. Features & Labels
X = news_dataset['content'].values
Y = news_dataset['label'].values

# 5. Train-Test Split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.5, stratify=Y, random_state=2
)

# 6. Vectorization 
vectorizer = TfidfVectorizer(
    max_features=500,
    max_df=0.9
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# 7. Model Selection
model = LogisticRegression(max_iter=1000, C=0.1)
model.fit(X_train, Y_train)

# 8. Evaluation
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Training Accuracy:", accuracy_score(Y_train, train_pred))
print("Testing Accuracy:", accuracy_score(Y_test, test_pred))

print("\nClassification Report:\n")
print(classification_report(Y_test, test_pred))

# 9. Prediction Example
X_new = X_test[5]
prediction = model.predict(X_new)

if prediction[0] == 0:
    print("Predicted: Real")
else:
    print("Predicted: Fake")

# Actual label
res = Y_test[5]

if res == 0:
    print("Actual: Real")
else:
    print("Actual: Fake")

# 10. Custom Prediction
def predict_news(text):
    text = preprocess(text)
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return "Real News" if pred == 0 else "Fake News"

print("\nCustom Test:",predict_news("Breaking: Government announces new economic reform"))

Training Accuracy: 0.9425809612900352
Testing Accuracy: 0.9399973272751571

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.94      0.94     10709
           1       0.95      0.94      0.94     11740

    accuracy                           0.94     22449
   macro avg       0.94      0.94      0.94     22449
weighted avg       0.94      0.94      0.94     22449

Predicted: Real
Actual: Real

Custom Test: Real News
